# Segmentação de uma pluma de fumaça com o SAM (Segment Anything Model)

Notebook para a atividade de segmentação de pluma de fumaça: gerar a máscara com o SAM, **avaliar** o resultado e **melhorar** código, prompts e apresentação da máscara.

**Como usar**
1. No Colab: *Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4)*.
2. Coloque sua imagem em `images/pluma.jpg` (ou deixe de fora e faça o upload quando o notebook pedir).
3. Rode as células em ordem. As células marcadas com **AJUSTE** têm coordenadas de exemplo que você deve trocar pelas da *sua* imagem (use a grade da imagem para ler as coordenadas).
4. Preencha a seção final de análise com o que você observou nos seus resultados.

**Roteiro**
- Etapa 0: configuração, imagem e modelo
- Etapa 1: máscara automática do SAM (linha de base) e diagnóstico
- Etapa 2: prompts (caixa, pontos positivos e negativos), com no mínimo três configurações
- Etapa 3: melhorias (contraste, refinamento iterativo, pós-processamento)
- Etapa 4: métricas e diagnóstico da imagem (iluminação, contraste, fumaça vs. nuvens)
- Etapa 5: análise escrita e exportação da máscara final

## Etapa 0: configuração

In [ ]:
# Instalação (Colab). Os pesos do SAM vêm da Meta; vit_b é o mais leve, vit_h o mais preciso.
!pip -q install git+https://github.com/facebookresearch/segment-anything.git opencv-python-headless matplotlib pandas
!wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
# Para usar o modelo maior, descomente a linha abaixo e mude MODEL_TYPE para "vit_h" na próxima célula:
# !wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
import os
from pathlib import Path
import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

MODEL_TYPE = "vit_b"   # "vit_b" (leve) | "vit_l" | "vit_h" (melhor)
CHECKPOINTS = {"vit_b": "sam_vit_b_01ec64.pth",
               "vit_l": "sam_vit_l_0b3195.pth",
               "vit_h": "sam_vit_h_4b8939.pth"}

IMAGE_PATH = "images/pluma.jpg"                  # sua imagem
REF_MASK_PATH = "images/pluma_referencia.png"    # OPCIONAL: máscara desenhada à mão (branco = fumaça)

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

In [ ]:
# ---------- Funções auxiliares ----------
def overlay(img, mask, color=(255, 0, 0), alpha=0.5):
    out = img.astype(np.float32).copy()
    m = mask.astype(bool)
    out[m] = (1 - alpha) * out[m] + alpha * np.array(color, np.float32)
    return out.astype(np.uint8)

def draw_prompts(ax, points=None, labels=None, box=None):
    if points is not None:
        pts, lab = np.array(points), np.array(labels)
        ax.scatter(pts[lab == 1, 0], pts[lab == 1, 1], c="lime", marker="*", s=220,
                   edgecolors="white", linewidths=1.2, label="ponto positivo")
        ax.scatter(pts[lab == 0, 0], pts[lab == 0, 1], c="red", marker="X", s=140,
                   edgecolors="white", linewidths=1.2, label="ponto negativo")
    if box is not None:
        x0, y0, x1, y1 = box
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="cyan", lw=2))

def panel(img, mask, title, points=None, labels=None, box=None, save=None):
    """Imagem original | máscara binária | máscara sobreposta + prompts."""
    fig, axs = plt.subplots(1, 3, figsize=(16, 5.5))
    axs[0].imshow(img);                                            axs[0].set_title("Imagem original")
    axs[1].imshow(mask.astype(np.uint8) * 255, cmap="gray", vmin=0, vmax=255); axs[1].set_title("Máscara binária")
    axs[2].imshow(overlay(img, mask)); draw_prompts(axs[2], points, labels, box); axs[2].set_title("Máscara sobreposta + prompts")
    for a in axs:
        a.axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if save:
        fig.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()

def show_with_grid(img, step=None):
    """Mostra a imagem com grade em pixels, para você ler as coordenadas dos prompts."""
    h, w = img.shape[:2]
    step = step or max(int(round(max(h, w) / 10, -1)), 10)
    fig, ax = plt.subplots(figsize=(10, max(10 * h / w, 3)))
    ax.imshow(img)
    ax.set_xticks(range(0, w, step)); ax.set_yticks(range(0, h, step))
    ax.grid(color="yellow", alpha=0.45, lw=0.6); ax.tick_params(labelsize=7)
    ax.set_title(f"Imagem {w}x{h} px: x para a direita, y para baixo")
    plt.show()

def iou(a, b):
    a, b = a.astype(bool), b.astype(bool)
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 1.0

def area_pct(m):
    return 100.0 * float(m.astype(bool).mean())

In [ ]:
# ---------- Imagem ----------
if not os.path.exists(IMAGE_PATH):
    from google.colab import files           # só existe no Colab
    up = files.upload()
    IMAGE_PATH = next(iter(up))

bgr = cv2.imread(IMAGE_PATH)
assert bgr is not None, f"Não consegui abrir {IMAGE_PATH}"
image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
H, W = image.shape[:2]
print("Tamanho da imagem:", W, "x", H)
show_with_grid(image)

In [ ]:
# ---------- Modelo ----------
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINTS[MODEL_TYPE])
sam.to(device=DEVICE)
predictor = SamPredictor(sam)

_current_img = {"ref": None}
def run_sam(img, box=None, points=None, labels=None, mask_input=None):
    """Roda o SAM com prompts. Devolve (melhor máscara, score, todas as máscaras, scores, logits da melhor)."""
    if _current_img["ref"] is not img:          # só recodifica a imagem quando ela muda
        predictor.set_image(img)
        _current_img["ref"] = img
    multi = box is None                         # sem caixa há mais ambiguidade: pede 3 candidatas
    masks, scores, logits = predictor.predict(
        point_coords=None if points is None else np.array(points),
        point_labels=None if labels is None else np.array(labels),
        box=None if box is None else np.array(box),
        mask_input=mask_input,
        multimask_output=multi)
    best = int(np.argmax(scores))
    return masks[best], float(scores[best]), masks, scores, logits[best][None]

## Etapa 1: máscara automática do SAM (linha de base)

O gerador automático propõe várias máscaras sem nenhum prompt. Observe **quais** delas cobrem a pluma e quais pegam céu, nuvens ou edifícios.

In [ ]:
mask_generator = SamAutomaticMaskGenerator(
    sam, points_per_side=32, pred_iou_thresh=0.86,
    stability_score_thresh=0.92, min_mask_region_area=200)
auto = sorted(mask_generator.generate(image), key=lambda d: d["area"], reverse=True)
print(len(auto), "máscaras geradas")

top = auto[:6]
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, d) in enumerate(zip(axs.ravel(), top)):
    ax.imshow(overlay(image, d["segmentation"], color=(255, 0, 0)))
    ax.set_title(f"#{i}  área={100*d['area']/(H*W):.1f}%  iou_pred={d['predicted_iou']:.2f}", fontsize=10)
    ax.axis("off")
plt.suptitle("Máscaras automáticas (ordenadas por área)")
plt.tight_layout(); plt.show()

In [ ]:
# AJUSTE: escolha o índice (#) da máscara automática que mais se aproxima da pluma
AUTO_IDX = 0
baseline = auto[AUTO_IDX]["segmentation"]
panel(image, baseline, f"Linha de base: máscara automática #{AUTO_IDX}", save=OUT / "00_baseline_automatica.png")
cv2.imwrite(str(OUT / "mask_00_baseline.png"), baseline.astype(np.uint8) * 255)

**Perguntas desta etapa** (responda na seção final): a máscara cobre a pluma? Inclui regiões que não pertencem a ela? Deixa partes da fumaça de fora?

## Etapa 2: prompts (caixa, pontos positivos e negativos)

**AJUSTE:** as coordenadas abaixo são apenas proporções da imagem para o notebook rodar de ponta a ponta. Troque pelas coordenadas reais lidas na grade da imagem.

- ponto **positivo** (label 1): em cima da fumaça, inclusive nas partes mais claras e finas
- ponto **negativo** (label 0): onde o SAM costuma errar (céu, nuvens, edifícios, solo)

In [ ]:
BOX = [int(.30 * W), int(.10 * H), int(.70 * W), int(.75 * H)]   # [x0, y0, x1, y1]
POS = [[int(.50 * W), int(.45 * H)], [int(.50 * W), int(.30 * H)]]  # pontos sobre a fumaça
NEG = [[int(.10 * W), int(.08 * H)], [int(.90 * W), int(.08 * H)]]  # céu/nuvens/edifícios

PROMPTS = {
    "A_caixa":        dict(box=BOX, points=None, labels=None),
    "B_pontos":       dict(box=None, points=POS + NEG, labels=[1] * len(POS) + [0] * len(NEG)),
    "C_caixa_pontos": dict(box=BOX, points=[POS[0]] + NEG[:1], labels=[1, 0]),
    # Crie outras variações (ex.: só pontos positivos, mais pontos negativos, caixa menor...)
}

In [ ]:
results = {}
for name, p in PROMPTS.items():
    mask, score, masks, scores, logit = run_sam(image, p["box"], p["points"], p["labels"])
    results[name] = dict(mask=mask, score=score, logit=logit, prompt=p)
    panel(image, mask, f"{name}  |  score SAM = {score:.3f}  |  área = {area_pct(mask):.1f}%",
          p["points"], p["labels"], p["box"], save=OUT / f"01_{name}.png")
    cv2.imwrite(str(OUT / f"mask_{name}.png"), mask.astype(np.uint8) * 255)
    if len(masks) > 1:    # prompts só com pontos: mostra as 3 candidatas do SAM
        fig, axs = plt.subplots(1, len(masks), figsize=(14, 4))
        for k, ax in enumerate(axs):
            ax.imshow(overlay(image, masks[k])); ax.set_title(f"candidata {k}  score={scores[k]:.3f}"); ax.axis("off")
        plt.suptitle(f"{name}: candidatas (multimask_output)"); plt.tight_layout(); plt.show()

## Etapa 3: melhorias

Três ideias para testar (e **justificar** na análise): (a) realçar o contraste antes do SAM, (b) refinar iterativamente com pontos de correção, (c) pós-processar a máscara.

In [ ]:
# (a) Realce de contraste (CLAHE no canal de luminosidade)
def enhance_contrast(img, clip=3.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

image_clahe = enhance_contrast(image)
p = PROMPTS["C_caixa_pontos"]
mask, score, _, _, logit = run_sam(image_clahe, p["box"], p["points"], p["labels"])
results["D_C_com_CLAHE"] = dict(mask=mask, score=score, logit=logit, prompt=p)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].imshow(image); axs[0].set_title("Original"); axs[1].imshow(image_clahe); axs[1].set_title("Após CLAHE")
for a in axs: a.axis("off")
plt.show()
panel(image, mask, f"D: prompts de C na imagem com CLAHE | score={score:.3f} | área={area_pct(mask):.1f}%",
      p["points"], p["labels"], p["box"], save=OUT / "02_D_clahe.png")
cv2.imwrite(str(OUT / "mask_D_clahe.png"), mask.astype(np.uint8) * 255)

In [ ]:
# (b) Refinamento iterativo: reaproveita a máscara anterior (logits) e adiciona pontos de correção
# AJUSTE: positivos onde a fumaça ficou de fora; negativos onde a máscara invadiu céu/nuvem/edifício
FIX_POINTS = [[int(.50 * W), int(.20 * H)], [int(.15 * W), int(.50 * H)]]
FIX_LABELS = [1, 0]

m1, s1, _, _, lg1 = run_sam(image, box=BOX)                               # passo 1: só a caixa
m2, s2, _, _, lg2 = run_sam(image, box=BOX, points=FIX_POINTS,
                            labels=FIX_LABELS, mask_input=lg1)            # passo 2: + correções
results["E_refinamento"] = dict(mask=m2, score=s2, logit=lg2,
                                prompt=dict(box=BOX, points=FIX_POINTS, labels=FIX_LABELS))
panel(image, m2, f"E: refinamento iterativo | score={s2:.3f} | área={area_pct(m2):.1f}%",
      FIX_POINTS, FIX_LABELS, BOX, save=OUT / "03_E_refinamento.png")
cv2.imwrite(str(OUT / "mask_E_refinamento.png"), m2.astype(np.uint8) * 255)

In [ ]:
# (c) Pós-processamento: fechamento/abertura morfológica, maior componente, preenchimento de buracos
# e (opcional) filtro de cor. Fumaça costuma ter saturação BAIXA; céu azul tem saturação alta.
# Cuidado: nuvens e fumaça branca/cinza passam no mesmo filtro. Teste e justifique.
def postprocess(mask, img=None, close_k=15, open_k=5, keep_largest=True,
                min_area_frac=0.002, fill_holes=True, sat_max=None, val_range=None):
    m = mask.astype(np.uint8)
    if close_k:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k)))
    if open_k:
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k)))
    if img is not None and (sat_max is not None or val_range is not None):
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        keep = np.ones(m.shape, bool)
        if sat_max is not None:
            keep &= hsv[..., 1] <= sat_max
        if val_range is not None:
            keep &= (hsv[..., 2] >= val_range[0]) & (hsv[..., 2] <= val_range[1])
        m = m & keep.astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        ids = [1 + int(np.argmax(areas))] if keep_largest else \
              [i + 1 for i, a in enumerate(areas) if a >= min_area_frac * m.size]
        m = np.isin(lab, ids).astype(np.uint8)
    if fill_holes:
        inv = (1 - m).astype(np.uint8)
        n2, lab2, _, _ = cv2.connectedComponentsWithStats(inv, connectivity=4)
        border = set(np.unique(np.concatenate([lab2[0], lab2[-1], lab2[:, 0], lab2[:, -1]])).tolist())
        holes = [i for i in range(1, n2) if i not in border]
        if holes:
            m[np.isin(lab2, holes)] = 1
    return m.astype(bool)

# AJUSTE os parâmetros (sat_max de 0 a 255; val_range = (mín, máx) de brilho de 0 a 255). None = não usar.
mask_pp = postprocess(results["E_refinamento"]["mask"], image, sat_max=None, val_range=None)
results["F_E_posprocessada"] = dict(mask=mask_pp, score=results["E_refinamento"]["score"], logit=None,
                                    prompt=results["E_refinamento"]["prompt"])
panel(image, mask_pp, f"F: E + pós-processamento | área={area_pct(mask_pp):.1f}%",
      FIX_POINTS, FIX_LABELS, BOX, save=OUT / "04_F_posprocessada.png")
cv2.imwrite(str(OUT / "mask_F_posprocessada.png"), mask_pp.astype(np.uint8) * 255)

## Etapa 4: métricas e diagnóstico da imagem

Sem uma máscara de referência não existe "acerto" absoluto. Aqui você tem duas opções: (1) comparar as configurações entre si (IoU) e (2) **desenhar à mão uma máscara de referência** (em qualquer editor de imagem, fumaça em branco e o resto em preto) e salvá-la em `images/pluma_referencia.png`. Com ela, o notebook calcula IoU, Dice, precisão e revocação.

In [ ]:
names = list(results)
rows = [dict(config=n, area_pct=round(area_pct(results[n]["mask"]), 1), score_SAM=round(results[n]["score"], 3)) for n in names]
df = pd.DataFrame(rows)

ref = None
if os.path.exists(REF_MASK_PATH):
    r = cv2.imread(REF_MASK_PATH, cv2.IMREAD_GRAYSCALE)
    ref = cv2.resize(r, (W, H), interpolation=cv2.INTER_NEAREST) > 127
    def prf(m):
        m = m.astype(bool); tp = (m & ref).sum()
        prec = tp / m.sum() if m.sum() else 0.0; rec = tp / ref.sum() if ref.sum() else 0.0
        dice = 2 * tp / (m.sum() + ref.sum()) if (m.sum() + ref.sum()) else 1.0
        return pd.Series(dict(IoU_ref=iou(m, ref), Dice=dice, precisao=prec, revocacao=rec))
    df = pd.concat([df, pd.DataFrame([prf(results[n]["mask"]) for n in names]).round(3)], axis=1)
else:
    print("Sem máscara de referência (", REF_MASK_PATH, "). Mostrando só área e score do SAM.")
display(df) if "display" in globals() else print(df.to_string(index=False))
df.to_csv(OUT / "metricas.csv", index=False)

# Matriz de IoU entre configurações
M = np.array([[iou(results[a]["mask"], results[b]["mask"]) for b in names] for a in names])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(M, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w" if M[i, j] < .6 else "k", fontsize=8)
plt.colorbar(im); ax.set_title("IoU entre configurações"); plt.tight_layout()
fig.savefig(OUT / "05_matriz_iou.png", dpi=150); plt.show()

In [ ]:
# Comparação lado a lado de todas as configurações
n = len(results); cols = 3; rows_ = int(np.ceil((n + 1) / cols))
fig, axs = plt.subplots(rows_, cols, figsize=(5 * cols, 4 * rows_))
axs = axs.ravel(); axs[0].imshow(image); axs[0].set_title("Original")
for ax, (k, r) in zip(axs[1:], results.items()):
    ax.imshow(overlay(image, r["mask"])); ax.set_title(f"{k}\n{area_pct(r['mask']):.1f}% da imagem", fontsize=9)
for ax in axs:
    ax.axis("off")
plt.tight_layout(); fig.savefig(OUT / "06_comparacao_geral.png", dpi=130); plt.show()

In [ ]:
# Diagnóstico: o que nesta imagem dificulta a segmentação?
CLOUD_BOX = None   # AJUSTE (opcional): [x0, y0, x1, y1] sobre uma região de NUVEM/céu claro, para medir a semelhança com a fumaça

def image_diagnostics(img, mask, ring=25, cloud_box=None):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    m = mask.astype(np.uint8)
    dil = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ring, ring)))
    ring_m = (dil == 1) & (m == 0)                                    # faixa logo fora da máscara
    grad = np.hypot(cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3))
    edge = cv2.morphologyEx(m, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
    inside = m == 1
    d = {
        "brilho_L_dentro": float(lab[..., 0][inside].mean()),
        "brilho_L_entorno": float(lab[..., 0][ring_m].mean()) if ring_m.any() else np.nan,
        "contraste_interno_(desvio de L)": float(lab[..., 0][inside].std()),
        "saturacao_media_dentro": float(hsv[..., 1][inside].mean()),
        "deltaE_dentro_vs_entorno": float(np.linalg.norm(lab[inside].mean(0) - lab[ring_m].mean(0))) if ring_m.any() else np.nan,
        "gradiente_na_borda": float(grad[edge].mean()) if edge.any() else np.nan,
        "gradiente_medio_da_imagem": float(grad.mean()),
    }
    d["razao_gradiente_borda/imagem"] = d["gradiente_na_borda"] / d["gradiente_medio_da_imagem"]
    if cloud_box is not None:
        x0, y0, x1, y1 = cloud_box
        reg = lab[y0:y1, x0:x1].reshape(-1, 3)
        d["deltaE_fumaca_vs_nuvem"] = float(np.linalg.norm(lab[inside].mean(0) - reg.mean(0)))
    return d

diag = pd.DataFrame({k: image_diagnostics(image, r["mask"], cloud_box=CLOUD_BOX) for k, r in results.items()}).T.round(2)
diag.to_csv(OUT / "diagnostico_imagem.csv")
display(diag) if "display" in globals() else print(diag.to_string())

**Como ler o diagnóstico (heurísticas, não regras):**
- `deltaE_dentro_vs_entorno` baixo (por exemplo, abaixo de ~10 a 15 no espaço Lab) significa cor muito parecida com o entorno, o que dificulta a borda.
- `razao_gradiente_borda/imagem` abaixo de 1 indica borda mais difusa que o restante da imagem (fumaça se dissolvendo no ar).
- `contraste_interno` alto com `saturacao` baixa sugere fumaça heterogênea, com partes claras e escuras que o SAM pode separar em máscaras diferentes.
- `deltaE_fumaca_vs_nuvem` baixo mostra o risco de a máscara vazar para as nuvens.

## Etapa 5: análise (preencha com os **seus** resultados)

**1. A máscara automática (linha de base) cobre a pluma? Inclui o que não é pluma? Deixa fumaça de fora?**

*Sua resposta:*

**2. Compare as configurações A, B e C (e D, E, F): qual funcionou melhor e por quê?** Use as figuras em `outputs/`, a tabela de métricas e a matriz de IoU.

*Sua resposta:*

**3. Quais características da imagem dificultaram a segmentação?**

- Iluminação (contraluz, sombra, hora do dia):
- Contraste (fumaça clara sobre céu claro, borda difusa):
- Semelhança entre fumaça e nuvens (use `deltaE_fumaca_vs_nuvem` e as imagens):
- Outras (edifícios, relevo, vegetação, resolução):

**4. Quais melhorias você fez e por que (código, prompts, apresentação)?** O CLAHE ajudou? O refinamento com pontos negativos removeu o que sobrava? O pós-processamento ajudou ou apagou fumaça fina?

*Sua resposta:*

**5. Limitações e próximos passos** (outros modelos, mais imagens, máscara de referência manual, SAM 2, etc.):

*Sua resposta:*

In [ ]:
# AJUSTE: escolha a configuração final e exporte máscara binária + sobreposição
FINAL = "F_E_posprocessada"
final_mask = results[FINAL]["mask"]
cv2.imwrite(str(OUT / "mascara_final.png"), final_mask.astype(np.uint8) * 255)
cv2.imwrite(str(OUT / "sobreposicao_final.png"), cv2.cvtColor(overlay(image, final_mask), cv2.COLOR_RGB2BGR))
panel(image, final_mask, f"Resultado final ({FINAL}) | área = {area_pct(final_mask):.1f}%", save=OUT / "resultado_final.png")
print("Arquivos em", OUT.resolve(), ":", sorted(p.name for p in OUT.iterdir()))